# Model Optimization and Quantization
## AIAT 122 – Deep Learning

## Learning objectives
- Convert a Keras model to TFLite and apply quantization (e.g. float16).
- Compare model size before and after quantization.

**Where is this used in real life?** Mobile and edge devices have limited memory and compute. **We use quantization (e.g. FP32 → FP16 or INT8) to shrink the model and speed up inference** instead of keeping full precision because many devices and runtimes are optimized for lower precision; a small accuracy trade-off often allows deployment where full FP32 would not fit.

**Prerequisites:** Basic TensorFlow/Keras. If TensorFlow import fails, see DOCS/COLAB_SETUP.md.

## Short theory
- **Quantization:** Store weights and activations in lower precision (FP16, INT8) instead of FP32.
- **Post-training quantization:** Convert an already-trained model; TFLite can do FP16 or dynamic-range INT8.
- **Benefits:** Smaller file size, faster inference, lower memory.
- **Trade-off:** Possible small accuracy drop; quantization-aware training can reduce it.

**📌 Covers slide(s):** None — Unit 5 (deployment) has no institution slides; use examples in file order.


## Inputs & Outputs
**Inputs:** TensorFlow, a small Keras model (built and trained here).  
**Dataset:** Synthetic — random data (no download; used to demonstrate quantization).  
**Outputs:** TFLite model files (default and float16), file sizes, and a sample inference. Run time: under ~3 min.


In [1]:
import numpy as np
import os
try:
    import tensorflow as tf
    print("TensorFlow version:", tf.__version__)
except Exception as e:
    err = str(e).lower()
    if "charset_normalizer" in err or "md__mypyc" in err or "partially initialized" in err:
        print("⚠️ Fix: pip install --upgrade charset-normalizer requests, then restart kernel.")
        raise RuntimeError("Fix: pip install --upgrade charset-normalizer requests, then restart kernel.") from e
    raise
print("✅ Imports OK.")

TensorFlow version: 2.13.0
✅ Imports OK.


### Step 1: Build and train a small model (2 epochs)

In [2]:
# Small model for demo; we use 2 epochs so it runs in ~1 min
X = np.random.randn(500, 10).astype(np.float32)
y = (X[:, 0] > 0).astype(np.float32).reshape(-1, 1)
model = tf.keras.Sequential([
    tf.keras.layers.Dense(32, activation="relu", input_shape=(10,)),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.fit(X, y, epochs=2, verbose=1)
print("✅ Model trained.")

Epoch 1/2
16/16 [==============================] - 0s 450us/step - loss: 0.8087 - accuracy: 0.4480
Epoch 2/2
16/16 [==============================] - 0s 340us/step - loss: 0.7574 - accuracy: 0.4960
✅ Model trained.


### Step 2: Convert to TFLite (default and float16) and compare sizes

In [3]:
# We use TFLite so the model can run on mobile/edge; quantization reduces size
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_default = converter.convert()
with open("model_default.tflite", "wb") as f:
    f.write(tflite_default)
size_default = os.path.getsize("model_default.tflite")

converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_fp16 = converter.convert()
with open("model_fp16.tflite", "wb") as f:
    f.write(tflite_fp16)
size_fp16 = os.path.getsize("model_fp16.tflite")

print(f"Default TFLite size: {size_default} bytes")
print(f"FP16 quantized size: {size_fp16} bytes")
print(f"Size ratio: {size_fp16/size_default:.2f}x")

INFO:tensorflow:Assets written to: /var/folders/7n/l2c2z2x57871xg4f_0drsv1m0000gn/T/tmpo_mo0dlo/assets


INFO:tensorflow:Assets written to: /var/folders/7n/l2c2z2x57871xg4f_0drsv1m0000gn/T/tmpo_mo0dlo/assets
2026-02-05 17:59:50.499636: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
2026-02-05 17:59:50.499746: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
2026-02-05 17:59:50.500285: I tensorflow/cc/saved_model/reader.cc:45] Reading SavedModel from: /var/folders/7n/l2c2z2x57871xg4f_0drsv1m0000gn/T/tmpo_mo0dlo
2026-02-05 17:59:50.500536: I tensorflow/cc/saved_model/reader.cc:91] Reading meta graph with tags { serve }
2026-02-05 17:59:50.500539: I tensorflow/cc/saved_model/reader.cc:132] Reading SavedModel debug info (if present) from: /var/folders/7n/l2c2z2x57871xg4f_0drsv1m0000gn/T/tmpo_mo0dlo
2026-02-05 17:59:50.501531: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:375] MLIR V1 optimization pass is not enabled
2026-02-05 17:59:50.501826: I tensorflow/cc/saved_model/load

INFO:tensorflow:Assets written to: /var/folders/7n/l2c2z2x57871xg4f_0drsv1m0000gn/T/tmp9fbmj7n4/assets


INFO:tensorflow:Assets written to: /var/folders/7n/l2c2z2x57871xg4f_0drsv1m0000gn/T/tmp9fbmj7n4/assets


Default TFLite size: 3196 bytes
FP16 quantized size: 3088 bytes
Size ratio: 0.97x


2026-02-05 17:59:50.704875: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
2026-02-05 17:59:50.704886: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
2026-02-05 17:59:50.704984: I tensorflow/cc/saved_model/reader.cc:45] Reading SavedModel from: /var/folders/7n/l2c2z2x57871xg4f_0drsv1m0000gn/T/tmp9fbmj7n4
2026-02-05 17:59:50.705277: I tensorflow/cc/saved_model/reader.cc:91] Reading meta graph with tags { serve }
2026-02-05 17:59:50.705280: I tensorflow/cc/saved_model/reader.cc:132] Reading SavedModel debug info (if present) from: /var/folders/7n/l2c2z2x57871xg4f_0drsv1m0000gn/T/tmp9fbmj7n4
2026-02-05 17:59:50.706204: I tensorflow/cc/saved_model/loader.cc:231] Restoring SavedModel bundle.
2026-02-05 17:59:50.720271: I tensorflow/cc/saved_model/loader.cc:215] Running initialization op on SavedModel bundle at path: /var/folders/7n/l2c2z2x57871xg4f_0drsv1m0000gn/T/tmp9fbmj7n4
2026-02-

In [4]:
# Run one inference with TFLite interpreter
interpreter = tf.lite.Interpreter(model_path="model_fp16.tflite")
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
interpreter.set_tensor(input_details[0]["index"], X[:1].astype(np.float32))
interpreter.invoke()
pred = interpreter.get_tensor(output_details[0]["index"])
print("Sample TFLite (FP16) prediction:", pred[0, 0])
print("After this cell you should see: FP16 model is often ~half the size of default.")

Sample TFLite (FP16) prediction: 0.5388745
After this cell you should see: FP16 model is often ~half the size of default.


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


## 🧩 Mini-exercise

**Try it:** Run inference with both the default and float16 TFLite models on 10 test samples and compare the outputs. Do they match exactly or only approximately?

---

## Summary
**What you did:** Built a small Keras model, converted it to TFLite (default and float16), and compared file sizes and ran a sample inference.

**In real life you'd also:** Use INT8 quantization for even smaller size, apply quantization-aware training to limit accuracy drop, and deploy the .tflite file on mobile or edge.

**The main idea:** Quantization reduces precision (e.g. FP32 → FP16) to shrink the model and speed up inference for deployment.

**Next:** `06_flask_fastapi_deployment.ipynb` shows how to serve a model via a REST API with Flask/FastAPI.